# Optimization under uncertainty: DSIVC

_What if we pumped differently?_

Everything up to here has answered a single question about a single plan: given how the supply well (`wellopt`) is scheduled to run, *how much* sulfate will the supplied water carry, and how sure are we? The history match sharpened that forecast distribution; the dataworth notebook asked which extra measurements would sharpen it further. None of it told us what to *do*.

This notebook closes the loop. The supply well is not a fixed object: we can throttle its extraction rate and we can choose *when* in the supply period to switch it on. Those are the levers a manager actually holds. We pose them as **decision variables** and search for the trade-off between supplying more water and keeping peak sulfate low — a bi-objective **optimization under uncertainty**, run entirely on the DSI emulator via the DSIVC workflow.

Where this sits in the sequence:

- [`../part1_05_dsi_basics/dizon_dsi_basics.ipynb`](../part1_05_dsi_basics/dizon_dsi_basics.ipynb) trained and conditioned the DSI emulator — we reuse that emulator and its conditioned posterior here.
- [`../part1_06_full_model_check/`](../part1_06_full_model_check/) validated the DSI posterior against the full model.
- [`../part1_07_dataworth/`](../part1_07_dataworth/) used the emulator to value held-back data.
- This notebook uses the same emulator to value held-back *decisions*.

> **Skeleton notebook.** The decision-variable training sweep this workflow depends on has not been baked yet (see [the sweep section](#The-training-sweep:-why-plain-DSI-is-not-enough) below for what it is and why it is needed). The code cells here are headed `TODO` and carry commented stub calls against the vendored `pyemu` `feat_dsivc` API. They are the intended shape of the workflow, not a runnable notebook. When `prebaked/` ships the sweep, the stubs become live code.

### Admin

We lean on the vendored dependency trees (`flopy`, `pyemu`) that ship with this repository. The DSIVC workflow lives in `pyemu.emulators` and is only present on the `rhugman/pyemu@feat_dsivc` branch we vendor — the asserts below fail loudly if a different `pyemu` is on the path. As elsewhere in the series, `herebedragons` (imported as `hbd`) holds the shared plotting and bookkeeping helpers, and every workspace this notebook creates lives *inside this notebook's own directory*.

In [ ]:
import os
import sys
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

import flopy
import pyemu
from pyemu.emulators import DSI, DSIVC

warnings.filterwarnings("ignore")

sys.path.insert(0, "..")
import herebedragons as hbd

Confirm we are running the vendored dependencies and not something else `pip` dragged in:

In [ ]:
assert "dependencies" in flopy.__file__, flopy.__file__
assert "dependencies" in pyemu.__file__, pyemu.__file__
# DSIVC only exists on the feat_dsivc branch we vendor
assert hasattr(pyemu.emulators, "DSIVC"), (
    "this pyemu has no DSIVC; refresh dependencies/pyemu from rhugman/pyemu@feat_dsivc"
)

**Prerequisite check.** DSIVC wraps an optimizer around the *already-fitted, already-conditioned* DSI emulator. From `part1_05_dsi_basics` we need two artifacts:

1. the fitted `DSI` emulator object, and
2. a conditioned (posterior) DSI observation ensemble.

Both come out of `part1_05_dsi_basics`. If that notebook has not been run, stop and run it first.

There is a third artifact DSIVC needs — a runstore-prepared DSI *template* (`dsi.pst`, `dsi.pickle`, `forward_run.py`) — but we cannot reuse `part1_05`'s `dsi_template/` for it. That template was built before the decision variables existed, so its `dsi.pst` carries no observations for them. DSIVC requires each decision variable to appear as a **zero-weight observation** in the inner `dsi.pst` (`prepare_pestpp` raises `decvars not found in inner dsi.pst observations` otherwise). So we rebuild the template *here*, from the training sweep, where the decision-variable columns are present — see [the fit step below](#How-DSIVC-works). The `part1_05` template still serves as the upstream existence check.

In [ ]:
# part1_05's template is the upstream existence check (and the source of the fitted
# emulator + posterior ensemble); it is NOT the inner template DSIVC uses here — that
# one is rebuilt from the sweep below, so its dsi.pst carries the decvar observations.
p15_t_d = Path("..") / "part1_05_dsi_basics" / "dsi_template"
if not (p15_t_d / "dsi.pst").exists():
    raise Exception(
        "you need to run the '../part1_05_dsi_basics/dizon_dsi_basics.ipynb' notebook "
        "first — it produces the fitted DSI emulator and conditioned posterior this "
        "notebook reuses"
    )

Worker count for the outer optimization. Each candidate plan triggers a *nested* PESTPP-IES conditioning of the emulator (more on that below), so this is parallelism over decisions, not over model runs. Physical cores only:

In [ ]:
num_workers = psutil.cpu_count(logical=False)
num_workers

## The decision problem

The decision is recast once, canonically, here. Two **decision variables** describe how the supply well is operated:

- a **rate multiplier** on the base supply-well extraction rate — how hard we pump, and
- a **switch-on day** within the supply period (308–728 d) — when we start.

Two **objectives** describe what we care about:

- **maximize the volume of water supplied** over the supply period, and
- **minimize P95(peak SO₄)** — the 95th percentile, across the posterior ensemble, of the peak sulfate concentration at the supply well.

These pull against each other. Pumping harder and earlier supplies more water but drags the redox front toward the well faster, raising peak sulfate; pumping gently and late keeps sulfate down but supplies less. There is no single best plan — there is a *front* of non-dominated trade-offs.

The objectives are **minimize-native**: lower P95(peak SO₄), more volume. No threshold is baked into the search, and none is needed — the front *is* the answer. Choosing P95 (rather than the mean) is the decision-support move: the manager is not asked "what is the expected peak?" but "how much water can I promise, and how high might the sulfate run while I do?" The percentile *is* the risk appetite, stated out loud — treatment capacity is sized to it.

A contractual trigger, if one exists, is an **illustrative lens** drawn across the same front, not a second analysis: with a 90 mg/L supply-contract trigger on the sulfate axis, the *reliable* (chance-constrained) reading is simply the segment of the front below the line — plans holding P95(peak SO₄) under 90 mg/L. (The EU drinking-water standard of 250 mg/L is comfortably met across the whole front; cost, not compliance, drives this decision.)

A unit note worth stating once. The forecast obs (`welopt-ly*`, variable `so4`) are carried in **mol/L** internally, the way the transport model writes them. A contractual trigger such as 90 mg/L is in mg/L. The conversion uses the molar mass of SO₄ (≈96.06 g/mol), so 90 mg/L ≈ `90e-3 / 96.06` mol/L. We convert the *trigger* into model units rather than the ensemble, so the optimization sees the same numbers everywhere.

In [ ]:
# TODO: pin the supply-period bookkeeping (no model run; constants only).
#
# SO4_MOLAR_MASS = 96.06            # g/mol
# CONTRACT_TRIGGER_MGL = 90.0       # illustrative supply-contract trigger (lens only)
# CONTRACT_TRIGGER_MOLL = CONTRACT_TRIGGER_MGL * 1e-3 / SO4_MOLAR_MASS
#
# SUPPLY_START, SUPPLY_END = 308.0, 728.0   # supply period (days); forecast lives here
# DECISION_DATE = 252.0                      # decision committed ~8 weeks before switch-on
#
# print(f"90 mg/L  ->  {CONTRACT_TRIGGER_MOLL:.4e} mol/L")

## The training sweep: why plain DSI is not enough

Here is the catch, and it is the whole reason this notebook needs its own prebaked artifact.

DSI is an **observation-space emulator**. It learned the joint distribution of the model outputs (the breakthrough series at every site and species, including the supply-well sulfate forecast) *as produced by the prior parameter ensemble running one fixed supply-well schedule*. Conditioning it with PESTPP-IES slides that distribution toward the data. At no point did the emulator see a run where the pump rate or switch-on day was different — because no such run exists in the prior Monte Carlo. The decision variables are **inputs to the forward model**, not outputs the emulator ever observed.

So the emulator cannot answer "what if we pumped differently?" by interpolation. It has no axis for the decision. Asking it to extrapolate a pumping change it never saw would be exactly the kind of unvalidated emulator use the [fidelity-check beat](../part1_05_dsi_basics/dizon_dsi_basics.ipynb) warned against.

The fix is to give the emulator that axis on purpose. **Coverage of decision space is designed, not inherited.** We run a dedicated **training sweep**: take the *posterior* parameter fields (the history-matched ensemble from `master_hm`), and for each one re-sample the decision variables across their bounds, then run the full model. Each run is a (posterior field, pumping plan) pair. The sweep thus spans both the residual parameter uncertainty *and* the decision space — and the decision variables now appear as columns in the training ensemble, alongside the outputs.

That is the sweep `prebaked/` will hold: roughly **200 full-model runs**, posterior fields crossed with resampled decision variables. At ~6 min per run on a MacBook, that sweep is about a day of wall-clock compute — paid once, by the maintainer, so the optimization below runs on the emulator in seconds. That cost asymmetry is the thesis of the whole series: full-model ensembles are too expensive to put inside an optimizer loop, so we emulate, but the emulator must first be *trained on the decisions it will be asked about*.

In [ ]:
# TODO: load the prebaked decision-variable training sweep (PENDING — not yet baked).
#
# What was (will be) run for you, and roughly what it cost:
#   ~200 full-model runs = posterior parameter fields (from master_hm) x decision
#   variables (rate multiplier + switch-on day) re-sampled across their bounds.
#   At ~6 min/run that is ~20 h of wall time on a MacBook, baked once by the maintainer.
#
# sweep_dir = Path("..") / ".." / "prebaked" / "dsivc_training_sweep"
# if not sweep_dir.exists():
#     raise Exception(
#         "the DSIVC training sweep has not been baked yet; this notebook is a skeleton. "
#         "See prebaked/ inventory in docs/REDESIGN.md."
#     )
#
# the sweep ensemble: rows = runs, columns = DSI observations PLUS the two decision-
# variable columns (rate multiplier, switch-on day) carried as extra observations.
# sweep = pd.read_csv(sweep_dir / "sweep_obs_ensemble.csv", index_col=0)
# sweep.shape

## How DSIVC works

**DSIVC** — "DSI variable control" — wraps an outer PESTPP-MOU optimization around the DSI emulator. Its central trick: the decision variables are treated as **observations** in DSI-world — columns of the training ensemble that a manager can *control* rather than merely observe. (This is why the sweep had to carry them as columns.)

Evaluating one candidate plan is a nested loop:

1. The outer optimizer (PESTPP-MOU) proposes a plan: specific values for the rate multiplier and switch-on day.
2. Those values are injected into the inner `dsi.pst` as **high-weight, zero-noise observation targets** — "condition the emulator on the world where we pump *this* way."
3. A full nested PESTPP-IES conditioning runs against the emulator in runstore mode (`pestpp-ies dsi.pst /e`), producing a conditioned posterior ensemble — the **"stack"**.
4. The stack is summarized into per-output **percentiles** (the **"stack stats"**). The P95 of peak sulfate is one of these.
5. Those stack-stats are the *outer* problem's observations; the objectives and constraints are defined on them.

So each outer model run is itself an ensemble conditioning — cheap, because it is the emulator, not the full model.

The constructor needs the fitted emulator, a runstore-prepared DSI template, and a posterior observation ensemble whose columns match the DSI observation names *exactly*. We rebuild the fitted emulator the same way `part1_05` did — same transform, same energy threshold — except the training data is now the **sweep**, so the decision-variable columns are part of the observation set.

Then we re-prepare the runstore template from this sweep-fitted emulator. This is the step that distinguishes the inner template from `part1_05`'s: `DSI.prepare_pestpp` writes one zero-weight observation per training-ensemble column, so a sweep-fitted emulator yields a `dsi.pst` that *contains the decision variables as zero-weight observations* — exactly the form DSIVC validates against. We point `dsi_t_d` at this fresh template, not at `part1_05`'s.

In [ ]:
# TODO: rebuild the fitted DSI emulator on the sweep ensemble (PENDING sweep).
#
# Mirror the part1_05 fit, but train on the sweep so the decvar columns are part of
# the observation set:
#
# transforms = [{"type": "standard_scaler"}]
# dsi = DSI(data=sweep, transforms=transforms, energy_threshold=0.975)
# dsi.fit()
#
# Re-prepare the runstore template FROM the sweep-fitted emulator. prepare_pestpp
# emits one zero-weight observation per training-ensemble column, so this dsi.pst
# carries the decvar columns as zero-weight observations — which DSIVC requires and
# part1_05's decvar-free template does not have. Build it inside THIS notebook dir;
# do NOT reuse part1_05/dsi_template.
#
# dsi_t_d = Path("dsi_template_sweep")     # fresh; carries the decvar observations
# dsi.prepare_pestpp(str(dsi_t_d), use_runstor=True)
#
# Sanity check: the decvars are now zero-weight obs in the inner dsi.pst.
# inner = pyemu.Pst(str(dsi_t_d / "dsi.pst")).observation_data
# assert all(d in inner.index for d in ["dv_rate_mult", "dv_switch_day"])
# assert (inner.loc[["dv_rate_mult", "dv_switch_day"], "weight"] == 0.0).all()

Load the conditioned posterior observation ensemble. DSIVC draws the decision-variable ranges and the stack-stats summary from this ensemble, so its columns must equal the **inner** DSI observation set *exactly* — including the decvar columns the sweep template added (the constructor checks this and complains if any column is missing or extra). The history-match signal still comes from `part1_05`'s conditioning; the decvar columns ride along from the sweep. DSI observation names are lowercase — the constructor will flag any that are not.

In [ ]:
# TODO: load the conditioned posterior DSI obs ensemble (PENDING).
#
# The history-matched posterior comes from part1_05's inner IES conditioning
# (dsi.<N>.obs.jcb, last iteration, written into part1_05/dsi_template/). But that
# ensemble has no decvar columns, while the constructor requires oe columns to EQUAL
# the sweep template's obs set (decvars included). So the posterior must be re-conditioned
# against the SWEEP-fitted emulator's dsi.pst before it can be handed to DSIVC — its
# columns then match dsi_t_d's observation set exactly.
#
# dsi_pst = pyemu.Pst(str(dsi_t_d / "dsi.pst"))   # the sweep template's pst
# post_iters = sorted(
#     int(f.split(".")[1]) for f in os.listdir(dsi_t_d)
#     if f.startswith("dsi.") and ".obs." in f and f.split(".")[1].isdigit()
# )
# last = max(post_iters)
# oe = pyemu.ObservationEnsemble.from_binary(
#     pst=dsi_pst, filename=str(dsi_t_d / f"dsi.{last}.obs.jcb")
# )
# oe.shape

## Naming the decision variables and the forecast

DSIVC needs the *observation names* of the two decision-variable columns, and we need the observation names that make up the forecast (peak SO₄ at the supply well over the supply period). The forecast obs follow the repository's canonical scheme — `oname:conc_otype:lst_usecol:sim_time:<T>_obsid:welopt-ly1_variable:so4` and likewise for `welopt-ly3`/`welopt-ly5` — spanning all three supply-well screens (the forecast maxes over them, the same definition used throughout the series) at simulation times in the supply window.

> **Open item for the maintainer.** The two decision-variable column names depend on how the sweep script names them when it bakes the ensemble. The stubs below assume `dv_rate_mult` and `dv_switch_day`; reconcile these with the actual sweep column headers once it ships.

In [ ]:
# TODO: name the decvar columns and assemble the forecast obs list (PENDING sweep).
#
# decvar_names = ["dv_rate_mult", "dv_switch_day"]   # reconcile with sweep headers
#
# Forecast obs: supply-well SO4 over the supply period, across ALL supply-well
# screens (welopt-ly1/ly3/ly5) -- the canonical forecast maxes over them.
# obs = dsi_pst.observation_data
# fore = obs[
#     (obs.obsid.isin(["welopt-ly1", "welopt-ly3", "welopt-ly5"]))  # all supply-well screens
#     & (obs.variable == "so4")
#     & (obs.time.astype(float) >= SUPPLY_START)
#     & (obs.time.astype(float) <= SUPPLY_END)
# ]
# forecast_obsnmes = fore.obsnme.tolist()
# len(forecast_obsnmes)

## Build the outer optimization interface

`DSIVC.prepare_pestpp` builds the outer PESTPP-MOU control file. We hand it a fresh template directory (it must differ from the DSI template, which it copies and never modifies), the decision-variable names, and the percentiles to summarize the stack with — we ask for the 5th, 50th, and 95th, because **P95** is the objective we want and the others are useful context.

`inner_noptmax` controls how many iterations the *nested* emulator conditioning runs each time a plan is evaluated; a small number (a few) is enough on the emulator. `mou_population_size` is the size of the outer decision-variable population the optimizer evolves.

In [ ]:
# TODO: construct the DSIVC interface and prepare the outer PESTPP-MOU files (PENDING).
#
# dsivc = DSIVC(emulator=dsi, dsi_t_d=str(dsi_t_d), oe=oe, verbose=True)
#
# mou_t_d = Path("dsivc_template")   # fresh; created inside THIS notebook dir
# pst_mou = dsivc.prepare_pestpp(
#     str(mou_t_d),
#     decvar_names=decvar_names,
#     percentiles=[0.05, 0.5, 0.95],   # P95 = the risk-appetite objective
#     inner_noptmax=3,                 # nested emulator conditioning depth
#     decvar_weight=100.0,             # high weight: condition hard on the chosen plan
#     mou_population_size=2 * len(decvar_names) * 10,
#     seed=358,
# )
# prepare_pestpp returns dsivc.pst with noptmax=0; we still must define the objectives.

`prepare_pestpp` leaves the objectives undefined on purpose — it cannot know which stack-stats matter to us. We now wire up the **bi-objective** problem on the stack-stats observations.

The stack-stats are named `<org_obsnme>_stat:<stat>` — so the P95 of each supply-well sulfate obs is `<forecast_obsnme>_stat:95%`. Our two objectives:

- **minimize** the worst (over the supply period and screened layers) P95 supply-well sulfate — this is P95(peak SO₄); and
- **maximize** volume supplied, which is a function of the rate multiplier and the active supply duration (switch-on day to 728 d).

Volume is a deterministic function of the two decision variables, so we can carry it as a decvar-derived objective rather than a stack-stat. The exact wiring (a tied observation, or a custom objective expression) is a maintainer decision once the sweep column units are known — flagged below.

In [ ]:
# TODO: define the bi-objective MOU problem on the stack-stats (PENDING).
#
# obs_mou = pst_mou.observation_data
#
# Objective 1 (minimize): P95 peak SO4 at the supply well over the supply period.
# The 'peak' is the max over supply-period times/layers; with DSIVC's per-obs stack
# stats, take the single worst P95 across the forecast obs as the scalar objective,
# OR add a derived 'max' observation in the sweep. Simplest stub: the worst-case obs.
# p95_names = [f"{o}_stat:95%" for o in forecast_obsnmes]
# assert all(n in obs_mou.index for n in p95_names)
#
# Objective 2 (maximize): volume supplied. PESTPP-MOU minimizes, so encode as a
# 'less_than'/'greater_than' obs group accordingly (negate volume to maximize).
#
# pst_mou.pestpp_options["mou_objectives"] = ",".join([peak_so4_obj, neg_volume_obj])
# obs_mou.loc[peak_so4_obj, "obgnme"] = "less_than_obj"   # minimize peak SO4
# obs_mou.loc[neg_volume_obj, "obgnme"] = "less_than_obj" # minimize (-volume) = maximize volume
# obs_mou.loc[[peak_so4_obj, neg_volume_obj], "weight"] = 1.0

Set the number of generations and write the outer control file. Each generation evaluates the whole decision-variable population, and each evaluation is a nested emulator conditioning — so even though no full model runs here, this is the most compute-hungry notebook in the series after the prebaked sweeps.

In [ ]:
# TODO: finalize and write the outer control file (PENDING).
#
# pst_mou.control_data.noptmax = 30    # outer MOU generations
# pst_mou.write(str(mou_t_d / "dsivc.pst"), version=2)

## Run the optimization

Launch PESTPP-MOU in parallel. The model command for every run is the generated `dsivc_forward_run.py`, which runs the nested `pestpp-ies dsi.pst /e` conditioning of the emulator. No full model runs occur here — the ~6 min/run cost was already paid, once, baking the training sweep.

In [ ]:
# TODO: run the outer MOU optimization in parallel (PENDING).
#
# m_d = Path("master_dsivc")
# pyemu.os_utils.start_workers(
#     str(mou_t_d),
#     "pestpp-mou",
#     "dsivc.pst",
#     num_workers=num_workers,
#     worker_root=".",
#     master_dir=str(m_d),
# )

PESTPP-MOU writes a decision-variable population (`dsivc.<gen>.dv_pop.csv`) and an objective/observation population (`dsivc.<gen>.obs_pop.csv`) for each generation. The final generation's non-dominated members *are* the trade-off front: each is a supply-well plan you cannot improve on one objective without sacrificing the other.

We plot it with volume supplied on one axis and P95(peak SO₄) on the other. The front itself is the deliverable; if a supply contract carries a 90 mg/L trigger, we can overlay it as an illustrative lens line.

In [ ]:
# TODO: load the final-generation populations and plot the Pareto front (PENDING).
#
# gens = sorted(
#     int(f.split(".")[1]) for f in os.listdir(m_d)
#     if f.startswith("dsivc.") and f.endswith(".obs_pop.csv") and f.split(".")[1].isdigit()
# )
# last_gen = max(gens)
# dv_pop = pd.read_csv(m_d / f"dsivc.{last_gen}.dv_pop.csv", index_col=0)
# obs_pop = pd.read_csv(m_d / f"dsivc.{last_gen}.obs_pop.csv", index_col=0)
#
# volume = obs_pop[volume_obj].values            # m^3 supplied
# p95_so4 = obs_pop[peak_so4_obj].values         # mol/L; P95 of peak supply-well SO4

If a contractual trigger applies, reading it across the front is one sentence of interpretation, not a re-run. With P95(peak SO₄) on the sulfate axis and an illustrative 90 mg/L line drawn, members **below** the line are the **reliable** (chance-constrained) plans — a 95% posterior chance of staying under the trigger — and the manager picks the point on that segment supplying the most water. The unconstrained trade-off and the reliable reading are the *same figure*; we never re-ran anything to get it.

In [ ]:
# TODO: plot the front; overlay the 90 mg/L contract trigger as an illustrative lens (PENDING).
#
# fig, ax = plt.subplots(figsize=(6, 5))
# ax.scatter(volume, p95_so4 * SO4_MOLAR_MASS / 1e-3, c="0.4", s=25,  # back to mg/L for display
#            label=f"trade-off front (gen {last_gen})")
# ax.axhline(CONTRACT_TRIGGER_MGL, color="r", ls="--",
#            label="90 mg/L contract trigger (illustrative)")
# ax.set_xlabel("volume supplied (m$^3$)")
# ax.set_ylabel("P95(peak SO$_4$) at supply well (mg/L)")
# ax.set_title("Supply-well operation under uncertainty")
# ax.legend()
# fig.tight_layout()

## Wrap-up

What this notebook bought us:

- It turned the forecast from a *verdict* on one fixed plan into a *menu* of plans, each with its uncertainty honestly carried through.
- It did so on the emulator. The only full-model cost was the one-off training sweep — the optimizer loop, with its nested conditionings, never touched the ~6 min/run model.
- The volume trade-off and any reliability reading (the 95% chance of staying under a contract trigger) come out of one front, one figure.

And the caveat that earns its keep: the emulator can only reason about decisions it was *trained on*. The front is trustworthy across the decision-variable bounds the sweep covered, and no further. Widening the bounds means a new sweep — and another fidelity check before the answer is believed. Same principle, all the way down: never trust an emulator you have not tested, on a question it has not seen.